In [ ]:
from time import sleep
from urllib.error import HTTPError

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.time import Time
from astropy.utils.data import conf
from astropy.wcs import WCS
from astroquery.ipac.irsa import Irsa
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation, PillowWriter
from reproject import reproject_interp

conf.remote_timeout = 120

In [ ]:
ra_deg = 304.693508808
dec_deg = 42.4436872991
ancho_arcmin = 6
banda = "SPHEREx-D1"

centro = SkyCoord(ra_deg, dec_deg, unit="deg", frame="icrs")
ancho_grados = (ancho_arcmin * u.arcmin).to_value(u.deg)
radio_busqueda = 1 * u.arcsec

In [ ]:
resultados = Irsa.query_sia(
    pos=(centro, radio_busqueda),
    collection=["spherex_dr1", "spherex_qr3"],
)

if len(resultados) == 0:
    raise ValueError("La búsqueda no devolvió imágenes para este centro.")

resultados.sort("t_min")
print("Imágenes encontradas:", len(resultados))

columnas = ["obs_collection", "obs_id", "energy_bandpassname", "t_min"]
vista = resultados[columnas][:8].copy()
vista["inicio_utc"] = Time(vista["t_min"], format="mjd", scale="utc").iso
display(vista)

In [ ]:
numero_imagenes = 20
indices_manual = None  # Por ejemplo: [0, 2, 5, 8]
intervalo_ms = 700     # Tiempo en pantalla de cada cuadro

candidatas = resultados[resultados["energy_bandpassname"] == banda].copy()
candidatas.sort("t_min")
vistos, filas_unicas = set(), []
for i, toma in enumerate(candidatas):
    clave = (str(toma["obs_id"]), str(toma["energy_bandpassname"]))
    if clave not in vistos:
        vistos.add(clave)
        filas_unicas.append(i)
tomas = candidatas[filas_unicas]
if len(tomas) < 3:
    raise ValueError(f"Solo hay {len(tomas)} exposiciones únicas de {banda}. Prueba otro detector o campo para un video de más de dos cuadros.")

catalogo = tomas[["obs_collection", "obs_id"]].copy()
catalogo["fila"] = np.arange(len(tomas))
catalogo["inicio_utc"] = Time(tomas["t_min"], format="mjd", scale="utc").iso
if indices_manual is not None:
    if any(not isinstance(i, (int, np.integer)) or i < 0 or i >= len(tomas) for i in indices_manual):
        raise ValueError("Los índices manuales deben ser enteros válidos del catálogo.")
    indices_video = sorted(set(indices_manual))
else:
    if numero_imagenes is not None and (not isinstance(numero_imagenes, int) or numero_imagenes < 3):
        raise ValueError("numero_imagenes debe ser un entero de al menos 3 o None.")
    cantidad = len(tomas) if numero_imagenes is None else min(numero_imagenes, len(tomas))
    indices_video = np.linspace(0, len(tomas) - 1, cantidad, dtype=int).tolist()
if len(indices_video) < 3:
    raise ValueError("Selecciona al menos tres exposiciones distintas.")
print(f"Disponibles: {len(tomas)} exposiciones únicas de {banda}.")
print(f"Seleccionadas para descargar: {len(indices_video)}.")
display(catalogo[["fila", "inicio_utc", "obs_id", "obs_collection"]][indices_video])

In [ ]:
def cargar_toma(toma):
    enlace = str(toma["access_url"])
    enlace_recorte = (
        f"{enlace}?center={centro.ra.deg},{centro.dec.deg}d&size={ancho_grados}"
    )
    origen = "Recorte de IRSA"

    try:
        archivo_abierto = fits.open(enlace_recorte, cache=True, memmap=False)
    except HTTPError as error:
        if error.code != 503:
            raise
        print("IRSA devolvió 503. Reintentando el recorte en 5 segundos…")
        sleep(5)
        try:
            archivo_abierto = fits.open(enlace_recorte, cache=True, memmap=False)
        except HTTPError as segundo_error:
            if segundo_error.code != 503:
                raise
            print("El recorte sigue dando 503. Abriendo la imagen completa…")
            archivo_abierto = fits.open(enlace, cache=True, memmap=False)
            origen = "Imagen completa"

    with archivo_abierto as archivo:
        imagen = archivo["IMAGE"]
        cabecera = imagen.header.copy()
        pixeles = imagen.data.copy()
        wcs_celeste = WCS(cabecera).celestial
        x, y = wcs_celeste.world_to_pixel(centro)

        # La distorsión del cielo ya se aplicó al calcular x, y.
        # Quitamos sus órdenes de la copia dedicada a la longitud de onda.
        cabecera_espectral = cabecera.copy()
        for clave in ["A_ORDER", "B_ORDER", "AP_ORDER", "BP_ORDER"]:
            cabecera_espectral.pop(clave, None)
        wcs_espectral = WCS(cabecera_espectral, fobj=archivo, key="W")
        longitud, ancho_banda = wcs_espectral.pixel_to_world(x, y)

        return {
            "pixeles": pixeles,
            "cabecera": cabecera,
            "cabecera_principal": archivo[0].header.copy(),
            "extensiones": [parte.name for parte in archivo],
            "wcs": wcs_celeste,
            "lambda_um": float(longitud.to_value(u.um)),
            "ancho_banda_um": float(ancho_banda.to_value(u.um)),
            "fecha": Time(toma["t_min"], format="mjd", scale="utc"),
            "obs_id": str(toma["obs_id"]),
            "origen": origen,
        }

In [ ]:
fotos, fallos = [], []
for paso, indice in enumerate(indices_video, start=1):
    print(f"Descargando {paso}/{len(indices_video)} · fila {indice}…", flush=True)
    try:
        foto = cargar_toma(tomas[indice])
        foto["fila_catalogo"] = indice
        fotos.append(foto)
    except Exception as error:
        fallos.append((indice, type(error).__name__, str(error)))
        print(f"  OMITIDA: {type(error).__name__}: {error}")
fotos.sort(key=lambda foto: foto["fecha"].mjd)
print(f"Descargadas: {len(fotos)}. Fallidas: {len(fallos)}.")
if len(fotos) < 3:
    raise RuntimeError("No se descargaron tres imágenes. Revisa los errores o selecciona otras filas.")

In [ ]:
for nombre, foto in enumerate(fotos, start=1):
    cabecera = foto["cabecera"]
    print(f"\nFoto {nombre} | {foto['fecha'].iso} UTC")
    print(f"Longitud de onda cerca del centro: {foto['lambda_um']:.4f} µm")
    print(f"Ancho de banda cerca del centro: {foto['ancho_banda_um']:.4f} µm")
    print("Versión de procesamiento:", foto["cabecera_principal"].get("VERSION"))

    for clave in ["OBSID", "DETECTOR", "NAXIS1", "NAXIS2", "BUNIT", "MJD-AVG"]:
        print(clave, "=", cabecera.get(clave, "(ausente)"))

# Quita el primer # de la línea siguiente para ver toda la primera cabecera.
# print(fotos[0]["cabecera"].tostring(sep="\n", padding=False))

In [ ]:
# 6,15 segundos de arco por píxel es una escala cercana a la de SPHEREx.
escala_arcsec = 6.15
lado_px = max(2, round(ancho_grados * 3600 / escala_arcsec))
paso_deg = ancho_grados / lado_px
forma = (lado_px, lado_px)

wcs_salida = WCS(naxis=2)
wcs_salida.wcs.ctype = ["RA---TAN", "DEC--TAN"]
wcs_salida.wcs.radesys = "ICRS"
wcs_salida.wcs.crval = [centro.ra.deg, centro.dec.deg]
wcs_salida.wcs.crpix = [(lado_px + 1) / 2, (lado_px + 1) / 2]
wcs_salida.wcs.cdelt = [-paso_deg, paso_deg]
wcs_salida.wcs.cunit = ["deg", "deg"]

def alinear(foto):
    datos, cobertura = reproject_interp(
        (foto["pixeles"], foto["wcs"]),
        wcs_salida,
        shape_out=forma,
    )
    datos[(cobertura <= 0) | ~np.isfinite(datos)] = np.nan
    return datos

cuadros, fotos_utiles = [], []
for numero, foto in enumerate(fotos, start=1):
    try:
        cuadro = alinear(foto)
    except Exception as error:
        print(f"Toma {numero} omitida al alinear: {error}")
        continue
    porcentaje = 100 * np.isfinite(cuadro).mean()
    print(f"Toma {numero}: {porcentaje:.1f} % del campo con datos")
    if porcentaje > 0:
        cuadros.append(cuadro)
        fotos_utiles.append(foto)
    else:
        print("  Omitida: no cubre el campo con datos válidos.")
fotos = fotos_utiles
if len(cuadros) < 3:
    raise RuntimeError("Quedaron menos de tres imágenes utilizables. Elige otras filas o un campo diferente.")
print(f"El video tendrá {len(cuadros)} cuadros.")

In [ ]:
valores = np.concatenate([cuadro[np.isfinite(cuadro)] for cuadro in cuadros])
vmin, vmax = np.percentile(valores, [5, 99])
if vmax <= vmin:
    vmax = vmin + 1
mapa = plt.colormaps["gray"].with_extremes(bad="#172747")

def titulo(foto):
    return f"{foto['fecha'].iso[:19]} UTC · ≈ {foto['lambda_um']:.3f} µm"

muestra = np.linspace(0, len(cuadros) - 1, min(6, len(cuadros)), dtype=int)
filas = int(np.ceil(len(muestra) / 3))
fig, ejes = plt.subplots(filas, 3, figsize=(15, 4 * filas), squeeze=False, layout="constrained")
for eje in ejes.flat:
    eje.set_visible(False)
for eje, indice in zip(ejes.flat, muestra):
    eje.set_visible(True)
    visual = eje.imshow(cuadros[indice], origin="lower", cmap=mapa, vmin=vmin, vmax=vmax)
    eje.set_title(f"Cuadro {indice + 1}/{len(cuadros)}\n" + titulo(fotos[indice]), fontsize=9)
    eje.set_xlabel("Columna")
    eje.set_ylabel("Fila")
fig.colorbar(visual, ax=[e for e in ejes.flat if e.get_visible()], label=fotos[0]["cabecera"].get("BUNIT", "Brillo"))
plt.show()

In [ ]:
fig_anim, eje_anim = plt.subplots(figsize=(6, 5), layout="constrained")
imagen_anim = eje_anim.imshow(
    cuadros[0], origin="lower", cmap=mapa, vmin=vmin, vmax=vmax
)
etiqueta = eje_anim.set_title(f"Cuadro 1/{len(cuadros)}\n" + titulo(fotos[0]))
eje_anim.set_xlabel("Columna en la cuadrícula común")
eje_anim.set_ylabel("Fila en la cuadrícula común")

def mostrar_cuadro(indice):
    imagen_anim.set_data(cuadros[indice])
    etiqueta.set_text(f"Cuadro {indice + 1}/{len(cuadros)}\n" + titulo(fotos[indice]))
    return imagen_anim, etiqueta

animacion = FuncAnimation(fig_anim, mostrar_cuadro, frames=len(cuadros), interval=intervalo_ms)
plt.close(fig_anim)
display(HTML(animacion.to_jshtml()))
guardar_gif = False
if guardar_gif:
    animacion.save("spherex_video_varias_imagenes.gif", writer=PillowWriter(fps=1000 / intervalo_ms))
    print("GIF guardado: spherex_video_varias_imagenes.gif")